# SO-101 MiniVLA — 단계별 시작
이 notebook은 학습된 모델이 아니라 구현/데이터/GPU 통과 여부를 확인하는 실행 순서다. 먼저 README를 읽고 프로젝트 루트에서 실행한다.

In [ ]:
import os, sys
from pathlib import Path
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/real2sim/workflow.py').is_file()), None)
assert ROOT is not None, '프로젝트 내부에서 notebook을 실행하세요'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'source/soarm101_lab'))
os.environ['PYTHONPATH'] = str(ROOT / 'source/soarm101_lab')
import torch
from soarm101_lab.learning.minivla import MiniVLAConfig, ActionHead
print(torch.__version__, 'CUDA:', torch.cuda.is_available())


## 1. 실제 기본 action head shape와 gradient 확인
더미 feature는 구조 검증용이며 pretrained semantic 능력을 대체하지 않는다.

In [ ]:
cfg = MiniVLAConfig()
head = ActionHead(cfg)
features = (torch.randn(1,100,2048), torch.ones(1,100,dtype=torch.bool),
            torch.zeros(1,100,2), torch.zeros(1,100,dtype=torch.long))
state = torch.zeros(1,6)
actions = torch.randn(1,16,6)
mask = torch.ones(1,16,dtype=torch.bool)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-4)
loss, metrics = head.loss(features, state, actions, mask)
loss.backward(); optimizer.step(); optimizer.zero_grad(set_to_none=True)
head.eval()
print("loss:", metrics, "output:", head.sample(features,state).shape)
print("trainable head parameters:", sum(p.numel() for p in head.parameters()))


## 2. 모든 회귀 테스트
CPU에서 data/mask/LoRA-gradient API를 확인한다. Qwen random tiny 테스트는 실제 2B/NF4의 메모리 시험이 아니다.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pytest","-q","tests/minivla"],check=True)


## 3. 실제 GPU profile
아래 셀의 RUN_GPU_PROFILE을 켜면 모델을 다운로드하고 실제 backbone으로 optimizer step을 실행한다. MuJoCo는 종료한 상태에서 Frozen부터 검사한다.

In [ ]:
RUN_GPU_PROFILE = False
if RUN_GPU_PROFILE:
    subprocess.run([sys.executable,"-m","soarm101_lab.learning.minivla.profile","--config","source/soarm101_lab/soarm101_lab/learning/minivla/configs/frozen_deterministic.json"],check=True)


## 4. 실데이터 → 통계 → Frozen 학습
이 파일들은 직접 준비해야 한다. DATA_CONTRACT.md를 따르며, 실제 파일이 준비된 뒤 RUN_TRAIN=True로 바꾼다.

In [ ]:
from pathlib import Path
MANIFEST = Path("data/so101/manifest.json")
STATS = Path("data/so101/statistics.json")
RUN_TRAIN = False
if RUN_TRAIN:
    from soarm101_lab.learning.minivla.data import fit_statistics
    assert MANIFEST.exists(), "실제 demonstration manifest를 준비하세요"
    fit_statistics(MANIFEST, STATS)
    subprocess.run([sys.executable,"-m","soarm101_lab.learning.minivla.train",
        "--config","source/soarm101_lab/soarm101_lab/learning/minivla/configs/frozen_deterministic.json", "--manifest",str(MANIFEST),
        "--stats",str(STATS),"--out","outputs/workflow/runs/minivla/frozen"],check=True)


## 5. Frozen 이후 QLoRA와 ACT
README 명령을 사용한다. 동일 split과 seed를 유지한다. simulation task success, 언어 변경에 따른 target selection을 함께 비교한다.

## 6. MuJoCo
자신의 SO-101 MJCF와 source/soarm101_lab/soarm101_lab/learning/minivla/configs/robot_mapping.template.json 및 cases.template.json의 null 값을 채운 뒤 README rollout 명령을 실행한다. 실제 robot assets는 이 패키지에 포함되어 있지 않다.